# 03 — Scratch MLP on dense features

3-layer feed-forward network trained on 130 dense features (handcrafted + TF-IDF cosines + SBERT cosines + LSA + PCA). 5-fold GroupKFold. Per-epoch train and val curves logged to W&B.

In [1]:
import subprocess, sys
def _p(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for pkg in ['wandb', 'pyarrow']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        _p(pkg)

In [2]:
import os
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
    OUTPUT_DIR = Path("/kaggle/working")
    PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
    FEATURES_DIR = OUTPUT_DIR / "features"
    FIGURES_DIR = OUTPUT_DIR / "figures"
else:
    ROOT = Path.cwd()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent
    DATA_DIR = ROOT / "data"
    PREDICTIONS_DIR = ROOT / "predictions"
    FEATURES_DIR = ROOT / "features"
    FIGURES_DIR = ROOT / "figures"

for p in [PREDICTIONS_DIR, FEATURES_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"IS_KAGGLE={IS_KAGGLE}")
print(f"DATA_DIR={DATA_DIR}")
print(f"PREDICTIONS_DIR={PREDICTIONS_DIR}")
print(f"FEATURES_DIR={FEATURES_DIR}")
print(f"FIGURES_DIR={FIGURES_DIR}")

IS_KAGGLE=False
DATA_DIR=c:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\data
PREDICTIONS_DIR=c:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\predictions
FEATURES_DIR=c:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\features
FIGURES_DIR=c:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\figures


In [3]:
import wandb
WANDB_ENTITY = "23f2001849-dl-genai-project"
WANDB_PROJECT = "23f2001849-t22026"
USE_WANDB = False
try:
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    else:
        wandb_key = os.environ.get("WANDB_API_KEY", "").strip()
    if wandb_key:
        wandb.login(key=wandb_key, relogin=True)
        USE_WANDB = True
        print("W&B login ok")
    else:
        print("W&B key not found, running without logging")
except Exception as e:
    print(f"W&B login failed: {e}. Running without logging.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Krishna\_netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B login ok


In [4]:
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.special import expit
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, log_loss)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OPTIONS = ['A', 'B', 'C', 'D', 'E']
NUM_CHOICES = 5
FOLDS = 5
print(f'device: {DEVICE}')

device: cpu


In [5]:
train_long = pd.read_parquet(FEATURES_DIR / 'train_long.parquet')
test_long = pd.read_parquet(FEATURES_DIR / 'test_long.parquet')
train_meta = pd.read_parquet(FEATURES_DIR / 'train_meta.parquet')
test_meta = pd.read_parquet(FEATURES_DIR / 'test_meta.parquet')

with open(FEATURES_DIR / 'feat_cols.json') as fh:
    feat_info = json.load(fh)
FEAT_COLS = feat_info['FEAT_COLS']
print(f'feature count: {len(FEAT_COLS)}')

feature count: 130


## MAP@3 helpers

In [6]:
def map3_score(pred_letters, true_letters):
    s = 0.0
    for pred, true in zip(pred_letters, true_letters):
        for k, L in enumerate(pred[:3]):
            if L == true:
                s += 1.0 / (k + 1)
                break
    return s / len(true_letters)

def scores_to_top3(grid):
    order = np.argsort(-grid, axis=1)
    return [[OPTIONS[i] for i in row[:3]] for row in order]

def long_to_grid(long_df, scores, ids_order):
    id_to_row = {q: i for i, q in enumerate(ids_order)}
    grid = np.zeros((len(ids_order), NUM_CHOICES))
    for i in range(len(long_df)):
        row = long_df.iloc[i]
        grid[id_to_row[row['id']], OPTIONS.index(row['letter'])] = scores[i]
    return grid

## Model

In [7]:
class FeatDS(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        if self.y is not None: return self.X[i], self.y[i]
        return self.X[i]

class ScratchMLP(nn.Module):
    def __init__(self, in_dim, hidden=128, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def train_one_fold(X_tr, y_tr, X_val, y_val, fold_idx, epochs=40, bs=256, lr=1e-3):
    set_seed()
    model = ScratchMLP(X_tr.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()
    tr_dl = DataLoader(FeatDS(X_tr, y_tr), batch_size=bs, shuffle=True)
    va_dl = DataLoader(FeatDS(X_val, y_val), batch_size=512, shuffle=False)
    tr_all_dl = DataLoader(FeatDS(X_tr, y_tr), batch_size=512, shuffle=False)

    best_probs = None; best_val_loss = float('inf')
    for epoch in range(epochs):
        model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()

        model.eval()
        with torch.no_grad():
            tr_logits = []
            for xb, _ in tr_all_dl:
                xb = xb.to(DEVICE)
                tr_logits.append(model(xb).cpu().numpy())
            tr_logits = np.concatenate(tr_logits)
            va_logits = []
            for xb, _ in va_dl:
                xb = xb.to(DEVICE)
                va_logits.append(model(xb).cpu().numpy())
            va_logits = np.concatenate(va_logits)

        tr_probs = expit(tr_logits)
        va_probs = expit(va_logits)
        tr_pred = (tr_probs > 0.5).astype(int)
        va_pred = (va_probs > 0.5).astype(int)

        m = {
            'epoch': epoch,
            'train_loss': log_loss(y_tr, tr_probs.clip(1e-7, 1 - 1e-7)),
            'val_loss': log_loss(y_val, va_probs.clip(1e-7, 1 - 1e-7)),
            'train_accuracy': accuracy_score(y_tr, tr_pred),
            'train_precision': precision_score(y_tr, tr_pred, zero_division=0),
            'train_recall': recall_score(y_tr, tr_pred, zero_division=0),
            'train_f1': f1_score(y_tr, tr_pred, zero_division=0),
            'val_accuracy': accuracy_score(y_val, va_pred),
            'val_precision': precision_score(y_val, va_pred, zero_division=0),
            'val_recall': recall_score(y_val, va_pred, zero_division=0),
            'val_f1': f1_score(y_val, va_pred, zero_division=0),
        }
        if USE_WANDB:
            wandb.log({f'fold_{fold_idx}/{k}': v for k, v in m.items() if k != 'epoch'},
                       step=epoch)

        if m['val_loss'] < best_val_loss:
            best_val_loss = m['val_loss']
            best_probs = va_probs.copy()

    return best_probs

## 5-fold training

In [8]:
if USE_WANDB:
    wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name='mlp',
                reinit=True, tags=['model:mlp', 'features:dense'],
                config={'model': 'scratch_mlp', 'hidden': 128, 'dropout': 0.3,
                         'epochs': 40, 'batch_size': 256, 'lr': 1e-3,
                         'folds': FOLDS, 'features': len(FEAT_COLS)},
                settings=wandb.Settings(_disable_stats=True))

X = train_long[FEAT_COLS].astype(np.float32).values
X_test = test_long[FEAT_COLS].astype(np.float32).values
y = train_long['is_correct'].astype(np.float32).values
folds = train_long['fold'].values

oof_mlp = np.zeros(len(train_long))
t0 = time.time()

for f in range(FOLDS):
    tr_mask = folds != f
    va_mask = folds == f
    mu = X[tr_mask].mean(axis=0)
    sd = X[tr_mask].std(axis=0).clip(min=1e-6)
    Xt = ((X[tr_mask] - mu) / sd).astype(np.float32)
    Xv = ((X[va_mask] - mu) / sd).astype(np.float32)
    val_probs = train_one_fold(Xt, y[tr_mask], Xv, y[va_mask], f)
    oof_mlp[va_mask] = val_probs
    print(f'fold {f} done')

elapsed = time.time() - t0
print(f'training time: {elapsed:.1f}s')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


fold 0 done


wandb: WARNING Tried to log to step 0 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 1 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 2 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 3 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 4 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/

fold 1 done


wandb: WARNING Tried to log to step 0 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 1 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 2 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 3 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 4 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/

fold 2 done


wandb: WARNING Tried to log to step 0 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 1 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 2 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 3 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 4 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/

fold 3 done


wandb: WARNING Tried to log to step 0 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 1 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 2 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 3 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 4 that is less than the current step 39. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/

fold 4 done
training time: 88.9s


## OOF-level final metrics

In [9]:
oof_pred = (oof_mlp > 0.5).astype(int)
true_letters = train_meta['answer'].tolist()
true_ids_order = train_meta['id'].tolist()
grid = long_to_grid(train_long, oof_mlp, true_ids_order)
top3 = scores_to_top3(grid)

final_metrics = {
    'accuracy': accuracy_score(y.astype(int), oof_pred),
    'precision': precision_score(y.astype(int), oof_pred, zero_division=0),
    'recall': recall_score(y.astype(int), oof_pred, zero_division=0),
    'f1': f1_score(y.astype(int), oof_pred, zero_division=0),
    'log_loss': log_loss(y.astype(int), oof_mlp.clip(1e-7, 1 - 1e-7)),
    'top1_acc': float(np.mean([t[0] == a for t, a in zip(top3, true_letters)])),
    'map3': map3_score(top3, true_letters),
    'training_time_sec': elapsed,
}
print('MLP OOF metrics:')
for k, v in final_metrics.items():
    print(f'  {k}: {v:.4f}')

if USE_WANDB:
    wandb.log({f'final/{k}': v for k, v in final_metrics.items()})
    wandb.finish()

MLP OOF metrics:
  accuracy: 0.8000
  precision: 0.0000
  recall: 0.0000
  f1: 0.0000
  log_loss: 0.4797
  top1_acc: 0.3960
  map3: 0.5610
  training_time_sec: 88.9249


final/accuracy,▁
final/f1,▁
final/log_loss,▁
final/map3,▁
final/precision,▁
final/recall,▁
final/top1_acc,▁
final/training_time_sec,▁
fold_0/train_accuracy,▁▁▂▃▃▄▆▆▇▇█▇▇███████████████████████████
fold_0/train_f1,▁▁▃▄▅▆▇▇▇███████████████████████████████
+48,...


## Full training and test predictions

In [10]:
mu_all = X.mean(axis=0)
sd_all = X.std(axis=0).clip(min=1e-6)
X_norm = ((X - mu_all) / sd_all).astype(np.float32)
X_test_norm = ((X_test - mu_all) / sd_all).astype(np.float32)

set_seed()
model_full = ScratchMLP(X_norm.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model_full.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.BCEWithLogitsLoss()
Xt = torch.tensor(X_norm).to(DEVICE)
yt = torch.tensor(y).to(DEVICE)

for epoch in range(30):
    model_full.train()
    perm = torch.randperm(len(Xt))
    for i in range(0, len(Xt), 256):
        idx = perm[i:i+256]
        opt.zero_grad()
        loss_fn(model_full(Xt[idx]), yt[idx]).backward()
        opt.step()

model_full.eval()
with torch.no_grad():
    test_logits = model_full(torch.tensor(X_test_norm).to(DEVICE)).cpu().numpy()
test_pred = expit(test_logits)

np.save(PREDICTIONS_DIR / 'oof_mlp.npy', oof_mlp)
np.save(PREDICTIONS_DIR / 'test_mlp.npy', test_pred)
with open(PREDICTIONS_DIR / 'mlp_metrics.json', 'w') as fh:
    json.dump(final_metrics, fh, indent=2)

print(f'saved oof_mlp.npy shape={oof_mlp.shape}')
print(f'saved test_mlp.npy shape={test_pred.shape}')

saved oof_mlp.npy shape=(10000,)
saved test_mlp.npy shape=(2500,)
